In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
os.chdir("..")

In [17]:
#load data

ratings = pd.read_csv("data/raw/ml-latest-small/ratings.csv")
movies = pd.read_csv("data/raw/ml-latest-small/movies.csv")

In [18]:
#check heads

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [19]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [20]:
#Only getting likes >= 4

pos = ratings[ratings["rating"] >= 4.0].copy()

In [21]:
#Reindex users/items to 0..N-1

user_map  = {u:i for i,u in enumerate(sorted(pos.userId.unique()))}
item_map  = {m:i for i,m in enumerate(sorted(pos.movieId.unique()))}
pos["user_idx"] = pos.userId.map(user_map)
pos["item_idx"] = pos.movieId.map(item_map)

In [13]:
print(pos)

        userId  movieId  rating   timestamp  user_idx  item_idx
0            1        1     4.0   964982703         0         0
1            1        3     4.0   964981247         0         2
2            1        6     4.0   964982224         0         4
3            1       47     5.0   964983815         0        41
4            1       50     5.0   964982931         0        43
...        ...      ...     ...         ...       ...       ...
100830     610   166528     4.0  1493879365       608      6126
100831     610   166534     4.0  1493848402       608      6127
100832     610   168248     5.0  1493850091       608      6144
100833     610   168250     5.0  1494273047       608      6145
100834     610   168252     5.0  1493846352       608      6146

[48580 rows x 6 columns]


In [23]:
#Per-user time split (80/20)

pos = pos.sort_values(["user_idx","timestamp"])

def split_user(df):
    n = len(df)
    k = max(1, int(0.2*n))
    return pd.Series([df.iloc[:-k], df.iloc[-k:]], index=["train","val"])
splits = pos.groupby("user_idx").apply(split_user,include_groups=False)
train = pd.concat(splits["train"].tolist()).reset_index(drop=True)
val   = pd.concat(splits["val"].tolist()).reset_index(drop=True)


In [27]:
#Save processed artifacts

#[TODO]